# Data validation — `cohort_features` v1

Phase D of the production workflow. Generates per-split TFDV statistics, infers a schema from **train only**, and validates **val / test** against that schema. Outputs are written to `artifacts/validation/` and reviewed inline below.

**Why this is structured oddly.** TFDV (`tensorflow-data-validation`) only publishes wheels for linux x86_64 + Python 3.9–3.11. This project's primary `.venv` is macOS arm64 + Python 3.12 (issue #1 in `docs/open_questions.md` pins MIMIC-IV v3.1, but the venv version is a separate Apple-Silicon practicality). To avoid forcing the rest of the project onto an x86 / 3.11 stack just for TFDV, we run TFDV in a small linux/amd64 Docker container:

```
docker/validation/                       <- image build context
├── Dockerfile                           <- python:3.11-slim + tfdv
├── requirements.txt
└── run_validation.py                    <- batch entrypoint
scripts/run_validation.sh                <- host-side wrapper
artifacts/validation/                    <- outputs (gitignored)
```

The notebook on the host (Python 3.12) does **not** `import tfdv`. It shells out to `scripts/run_validation.sh`, then reads back the artifacts:

| Artifact | What it is |
|---|---|
| `schema.pbtxt` | Schema inferred from TRAIN (single source of truth for downstream contracts) |
| `{train,val,test}_stats.pb` | Per-split `DatasetFeatureStatisticsList` protos |
| `{val,test}_anomalies.pbtxt` | Anomalies found when validating each split against the TRAIN schema |
| `train_vs_{val,test}.html` | Facets Overview comparison views, rendered standalone |
| `summary.json` | Row counts, feature names, anomaly counts (machine-readable) |

**Prereqs:** Docker Desktop running, `gcloud auth application-default login` completed.

**Imputation policy** (open question #9) is intentionally out of scope here. This phase establishes the schema + drift contract; the imputation lookup is decided after we see the anomaly report.

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

_HERE = Path(os.path.abspath("__file__")).parent
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

from src import config

ARTIFACTS_DIR = _HERE / "artifacts" / "validation"
RUNNER        = _HERE / "scripts" / "run_validation.sh"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"features_table : {config.FEATURES_TABLE}")
print(f"artifacts dir  : {ARTIFACTS_DIR}")
print(f"runner script  : {RUNNER}")

## Run TFDV (containerized)

Builds the image (idempotent — first run pulls TF + Beam and takes ~5–10 min; subsequent runs use the build cache) and executes the batch validation. Streams logs into the cell output. Re-run this cell whenever `cohort_features` is rebuilt.

In [ ]:
env = os.environ.copy()
env["BQ_PROJECT"]     = config.PROJECT_ID
env["FEATURES_TABLE"] = config.FEATURES_TABLE

proc = subprocess.run(
    ["bash", str(RUNNER)],
    env=env,
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n" + proc.stderr, file=sys.stderr)
    raise RuntimeError(f"validation runner exited with code {proc.returncode}")

## Summary

Row counts per split, feature count, and anomaly counts. Anomalies > 0 do **not** automatically fail the notebook — they're surfaced for review in the next cell.

In [ ]:
summary = json.loads((ARTIFACTS_DIR / "summary.json").read_text())

print(f"features_table : {summary['features_table']}")
print(f"n_columns      : {summary['n_columns']}")
print("row_counts     :")
for split, n in summary["row_counts"].items():
    print(f"  {split:<5}  {n:>8,}")
print("anomaly_counts :")
for split, n in summary["anomaly_counts"].items():
    print(f"  {split:<5}  {n}")

## Anomalies — VAL and TEST vs TRAIN schema

Parses the human-readable `anomalies.pbtxt` files into a DataFrame. Each row is one (split, feature, anomaly type) triple. Expected anomalies for this cohort:

* Vitals + severity features may show coverage drift between splits (ICU rate isn't exactly constant across chronological folds).
* Rare categorical levels (e.g. uncommon `discharge_location`) may appear in val/test but not in train.

Real concerns to flag would be: missing feature in val/test, schema-type mismatch, or large drift on a structural feature (label, split, demographics).

In [ ]:
# Minimal pbtxt parser: we only need (feature_name, anomaly description).
# Avoids importing the tfdv/tensorflow_metadata proto bindings on the host.
_BLOCK_RE       = re.compile(r"anomaly_info\s*\{(.*?)\n\}\n", re.DOTALL)
_KEY_RE         = re.compile(r'key:\s*"([^"]+)"')
_SHORT_DESC_RE  = re.compile(r'short_description:\s*"([^"]+)"')
_DESCRIPTION_RE = re.compile(r'description:\s*"([^"]+)"')


def parse_anomalies(path: Path, split: str) -> list[dict]:
    if not path.exists():
        return []
    text = path.read_text()
    rows = []
    # `anomaly_info` is a map<string, AnomalyInfo>; the textproto form is
    # `anomaly_info { key: "feat" value { ... } }` repeated.
    for block in re.finditer(r"anomaly_info\s*\{(.+?)\n\}\n", text, re.DOTALL):
        body = block.group(1)
        key   = _KEY_RE.search(body)
        short = _SHORT_DESC_RE.search(body)
        desc  = _DESCRIPTION_RE.search(body)
        rows.append({
            "split":   split,
            "feature": key.group(1)  if key  else "",
            "type":    short.group(1) if short else "",
            "detail":  desc.group(1)  if desc  else "",
        })
    return rows


rows = (
    parse_anomalies(ARTIFACTS_DIR / "val_anomalies.pbtxt",  "val")
    + parse_anomalies(ARTIFACTS_DIR / "test_anomalies.pbtxt", "test")
)
anomalies_df = pd.DataFrame(rows, columns=["split", "feature", "type", "detail"])

if anomalies_df.empty:
    print("No anomalies reported.")
else:
    print(f"{len(anomalies_df)} anomalies across val + test:")
    display(anomalies_df)

## Facets Overview — TRAIN vs VAL

Interactive per-feature histograms with split-to-split overlay. Use the controls in the widget to sort by missing rate / mean / std / Chi-square distance to spot drift quickly. Same view for TRAIN vs TEST is in the next cell.

In [ ]:
display(HTML((ARTIFACTS_DIR / "train_vs_val.html").read_text()))

## Facets Overview — TRAIN vs TEST

In [ ]:
display(HTML((ARTIFACTS_DIR / "train_vs_test.html").read_text()))

## Schema artifact

The inferred-from-train schema is the contract every later phase reads: imputation logic (open question #9), transforms in the modeling notebooks, and the serving-time validator. We print the first chunk inline as a sanity check; the full file is `artifacts/validation/schema.pbtxt`.

In [ ]:
schema_text = (ARTIFACTS_DIR / "schema.pbtxt").read_text()
lines = schema_text.splitlines()
print(f"schema.pbtxt — {len(lines):,} lines, {len(schema_text):,} bytes")
print("--- first 80 lines ---")
print("\n".join(lines[:80]))